# 08 - Train best ClinVar CNN 601bp

Notebook này đóng gói model tốt nhất hiện tại của project:

- Window: `601 bp`
- Input: `ref + alt + mutation marker` 9-channel, cộng `8` kênh sinusoidal positional encoding
- Split: gene-group split
- Model: dilated 1D CNN + 1D local window attention
- Augmentation: reverse-complement augment khi train
- Evaluation: reverse-complement TTA
- Hard-negative mining: 2 epoch

Kết quả đã có trước đó:

- Test ROC-AUC: `0.9589`
- Test PR-AUC: `0.8507`
- F1 @ 0.5: `0.7410`
- Test F1 tại threshold chọn từ validation F1: `0.7543`

Notebook này không sửa `notebooks/01_read_clinvar_variant_summary.ipynb`.

## 1. Config

Mặc định `RUN_TRAIN = False` để tránh vô tình train lại và ghi đè artifact. Đổi thành `True` khi muốn train lại từ đầu.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
PROJECT_DIR = PROJECT_DIR.resolve()

CONFIG = {
    "length": 601,
    "architecture": "dilated_window_attention",
    "positional_encoding": "sinusoidal",
    "positional_dim": 8,
    "batch_size": 256,
    "epochs": 5,
    "hard_negative_epochs": 2,
    "learning_rate": 1e-3,
    "hard_negative_lr": 3e-4,
    "weight_decay": 1e-4,
    "random_state": 42,
    "num_workers": 0,
    "rc_augment": True,
    "rc_tta": True,
}

RUN_TRAIN = False
FORCE_OVERWRITE = False

SAFE_NAME = "cnn_gene_group_positional_encoding_601_dilated_window_attention_rcaug_rctta"
PROCESSED_DIR = PROJECT_DIR / "processed"
MODEL_DIR = PROJECT_DIR / "models"
SCRIPT_PATH = PROJECT_DIR / "scripts" / "train_cnn_gene_group_positional_1001.py"

paths = {
    "x": PROCESSED_DIR / "X_ref_alt_marker_601.npy",
    "y": PROCESSED_DIR / "y_601.npy",
    "metadata": PROCESSED_DIR / "clinvar_training_metadata_601.parquet",
    "model": MODEL_DIR / f"clinvar_{SAFE_NAME}_pytorch.pt",
    "metrics": PROCESSED_DIR / f"{SAFE_NAME}_metrics.json",
    "history": PROCESSED_DIR / f"{SAFE_NAME}_training_history_pytorch.parquet",
    "predictions": PROCESSED_DIR / f"{SAFE_NAME}_test_predictions_pytorch.parquet",
    "thresholds": PROCESSED_DIR / f"{SAFE_NAME}_threshold_table_pytorch.parquet",
    "val_thresholds": PROCESSED_DIR / f"{SAFE_NAME}_val_threshold_table_pytorch.parquet",
}

print("Project:", PROJECT_DIR)
print("Train script:", SCRIPT_PATH)
for name, path in paths.items():
    print(f"{name:12s}", path, "exists=", path.exists())

## 2. Train lại best model

Cell này gọi đúng script training hiện tại với cấu hình best. Nếu artifact đã tồn tại và `FORCE_OVERWRITE=False`, script sẽ chặn để tránh ghi đè.

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    "--project-dir", str(PROJECT_DIR),
    "--length", str(CONFIG["length"]),
    "--architecture", CONFIG["architecture"],
    "--positional-encoding", CONFIG["positional_encoding"],
    "--positional-dim", str(CONFIG["positional_dim"]),
    "--epochs", str(CONFIG["epochs"]),
    "--hard-negative-epochs", str(CONFIG["hard_negative_epochs"]),
    "--batch-size", str(CONFIG["batch_size"]),
    "--learning-rate", str(CONFIG["learning_rate"]),
    "--hard-negative-lr", str(CONFIG["hard_negative_lr"]),
    "--weight-decay", str(CONFIG["weight_decay"]),
    "--random-state", str(CONFIG["random_state"]),
    "--num-workers", str(CONFIG["num_workers"]),
]
if CONFIG["rc_augment"]:
    cmd.append("--rc-augment")
if CONFIG["rc_tta"]:
    cmd.append("--rc-tta")
if FORCE_OVERWRITE:
    cmd.append("--force")

print(" ".join(cmd))

if RUN_TRAIN:
    env = os.environ.copy()
    env.setdefault("TQDM_DISABLE", "0")
    subprocess.run(cmd, check=True, env=env)
else:
    print("RUN_TRAIN=False, skip training. Set RUN_TRAIN=True to execute.")

## 3. Đọc metrics đã lưu

Dùng cell này để kiểm tra artifact của best model hiện tại hoặc kết quả sau khi train lại.

In [ ]:
import pandas as pd

with open(paths["metrics"], "r", encoding="utf-8") as f:
    metrics = json.load(f)

summary = {
    "model_name": metrics["model_name"],
    "length": metrics["length"],
    "architecture": metrics["architecture"],
    "input_channels": metrics["input_channels"],
    "positional_encoding": metrics.get("positional_encoding", "sinusoidal"),
    "positional_dim": metrics["positional_dim"],
    "rc_augment": metrics["rc_augment"],
    "rc_tta": metrics["rc_tta"],
    "best_val_pr_auc": metrics["best_val_pr_auc"],
    "test_roc_auc": metrics["test_metrics"]["roc_auc"],
    "test_pr_auc": metrics["test_metrics"]["pr_auc"],
    "test_accuracy": metrics["test_metrics"]["accuracy"],
    "test_f1_at_05": metrics["test_f1_pathogenic_05"],
    "test_precision_at_05": metrics["test_precision_pathogenic_05"],
    "test_recall_at_05": metrics["test_recall_pathogenic_05"],
    "best_val_f1_threshold": metrics["best_val_f1_threshold"]["threshold"],
    "test_f1_at_best_val_f1_threshold": metrics["test_at_best_val_f1_threshold"]["f1_pathogenic"],
}

pd.Series(summary)

In [ ]:
history = pd.read_parquet(paths["history"])
history[[
    "phase", "epoch",
    "train_pr_auc", "val_pr_auc",
    "train_roc_auc", "val_roc_auc",
    "val_accuracy",
]]

## 4. Load checkpoint để inference

Cell này rebuild kiến trúc `DilatedClinVarCNN`, load checkpoint tốt nhất, và chuẩn bị helper predict theo row index trong dataset `601bp`.

In [ ]:
import importlib.util
import numpy as np
import torch
from torch import nn

spec = importlib.util.spec_from_file_location("train_cnn_module", SCRIPT_PATH)
train_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_mod)

setattr(torch.serialization, "add_safe_globals", getattr(torch.serialization, "add_safe_globals", lambda *_args, **_kwargs: None))

x = np.load(paths["x"], mmap_mode="r")
y = np.load(paths["y"], mmap_mode="r")
metadata = pd.read_parquet(paths["metadata"])

positional_encoding = train_mod.make_position_encoding(
    CONFIG["positional_encoding"], x.shape[1], CONFIG["positional_dim"]
)
input_channels = x.shape[2] + positional_encoding.shape[1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = train_mod.make_model(CONFIG["architecture"], input_channels).to(device)
checkpoint = torch.load(paths["model"], map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("device:", device)
print("x shape:", x.shape, x.dtype)
print("input_channels:", input_channels)
print("checkpoint model:", checkpoint["config"]["model_name"])

In [ ]:
def predict_dataset_indices(indices, batch_size=512, threshold=None, use_rc_tta=True):
    indices = np.asarray(indices, dtype=np.int64)
    if threshold is None:
        threshold = metrics["best_val_f1_threshold"]["threshold"]

    loader = train_mod.make_loader(
        x, y, indices, positional_encoding, batch_size=batch_size, num_workers=0
    )
    rc_loader = None
    if use_rc_tta:
        rc_loader = train_mod.make_loader(
            x, y, indices, positional_encoding, batch_size=batch_size, num_workers=0, reverse_complement=True
        )
    criterion = nn.BCEWithLogitsLoss()
    _, probs, labels = train_mod.run_eval_epoch(
        model, loader, rc_loader, criterion, device, "predict", use_rc_tta
    )
    out = metadata.iloc[indices].copy()
    out["y_true"] = labels.astype(int)
    out["pred_proba_pathogenic"] = probs
    out["pred_label"] = (probs >= threshold).astype(int)
    out["threshold"] = float(threshold)
    return out

# Example: predict first 8 rows from the dataset.
predict_dataset_indices(np.arange(8), batch_size=8)[[
    "GeneSymbol", "ClinicalSignificance", "Y", "y_true",
    "pred_proba_pathogenic", "pred_label", "threshold"
]]

## 5. Đọc test predictions đã lưu

Cell này tiện cho error analysis mà không cần chạy lại model.

In [ ]:
predictions = pd.read_parquet(paths["predictions"])
print(predictions.shape)
predictions[[
    "GeneSymbol", "ClinicalSignificance", "Y", "y_true",
    "pred_proba_pathogenic", "pred_label"
]].head()

In [ ]:
best_threshold = metrics["best_val_f1_threshold"]["threshold"]
error_view = predictions.assign(
    pred_label_best_val_f1=lambda df: (df["pred_proba_pathogenic"] >= best_threshold).astype(int),
    abs_error=lambda df: (df["pred_proba_pathogenic"] - df["y_true"]).abs(),
)
error_view.sort_values("abs_error", ascending=False)[[
    "GeneSymbol", "ClinicalSignificance", "ReferenceAlleleVCF", "AlternateAlleleVCF",
    "y_true", "pred_proba_pathogenic", "pred_label_best_val_f1", "abs_error"
]].head(20)